# House Price Prediction

This notebook implements a complete house price prediction workflow: dataset collection, exploratory data analysis, data cleaning, feature engineering, and feature selection. The code expects `housing.csv` to be present in the same folder.

## 1. Dataset Collection

Load the housing dataset from the local CSV file and verify the initial structure.

In [1]:
import pandas as pd
from pathlib import Path

DATA_PATH = Path('housing.csv')
df = pd.read_csv(DATA_PATH)

print('Dataset path:', DATA_PATH.resolve())
print('Rows:', df.shape[0])
print('Columns:', df.shape[1])
df.head(10)

Dataset path: C:\Users\G\Desktop\Languages\AI-MLMODELS\House_Price_Prediction\housing.csv
Rows: 545
Columns: 13


,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
1,12250000,8960,4,4,4,yes,no,no,no,yes,3,no,furnished
2,12250000,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
3,12215000,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished
4,11410000,7420,4,1,2,yes,yes,yes,no,yes,2,no,furnished
5,10850000,7500,3,3,1,yes,no,yes,no,yes,2,yes,semi-furnished
6,10150000,8580,4,3,4,yes,no,no,no,yes,2,yes,semi-furnished
7,10150000,16200,5,3,2,yes,no,no,no,no,0,no,unfurnished
8,9870000,8100,4,1,2,yes,yes,yes,no,yes,2,yes,furnished
9,9800000,5750,3,2,4,yes,yes,no,no,yes,1,yes,unfurnished


## 2. Exploratory Data Analysis (EDA)

Inspect data types, missing values, duplicates, and basic statistics to understand the dataset.

In [2]:
print('Column names:')
print(df.columns.tolist())

print('Data types:')
print(df.dtypes)

print('Missing values by column:')
print(df.isna().sum().sort_values(ascending=False))

print('Numeric feature summary:')
display(df.describe(include='number').transpose())

print('Categorical feature summary:')
display(df.describe(include='object').transpose())

print('Duplicate rows:')
print(df.duplicated().sum())

Column names:
['price', 'area', 'bedrooms', 'bathrooms', 'stories', 'mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'parking', 'prefarea', 'furnishingstatus']
Data types:
price                int64
area                 int64
bedrooms             int64
bathrooms            int64
stories              int64
mainroad            object
guestroom           object
basement            object
hotwaterheating     object
airconditioning     object
parking              int64
prefarea            object
furnishingstatus    object
dtype: object
Missing values by column:
price               0
area                0
bedrooms            0
bathrooms           0
stories             0
mainroad            0
guestroom           0
basement            0
hotwaterheating     0
airconditioning     0
parking             0
prefarea            0
furnishingstatus    0
dtype: int64
Numeric feature summary:


,count,mean,std,min,25%,50%,75%,max
price,545.0,4.766729e+06,1.870440e+06,1750000.0,3430000.0,4340000.0,5740000.0,13300000.0
area,545.0,5.150541e+03,2.170141e+03,1650.0,3600.0,4600.0,6360.0,16200.0
bedrooms,545.0,2.965138e+00,7.380639e-01,1.0,2.0,3.0,3.0,6.0
bathrooms,545.0,1.286239e+00,5.024696e-01,1.0,1.0,1.0,2.0,4.0
stories,545.0,1.805505e+00,8.674925e-01,1.0,1.0,2.0,2.0,4.0
parking,545.0,6.935780e-01,8.615858e-01,0.0,0.0,0.0,1.0,3.0


Categorical feature summary:


,count,unique,top,freq
mainroad,545,2,yes,468
guestroom,545,2,no,448
basement,545,2,no,354
hotwaterheating,545,2,no,520
airconditioning,545,2,no,373
prefarea,545,2,no,417
furnishingstatus,545,3,semi-furnished,227


Duplicate rows:
0


## 3. Data Cleaning

Clean the dataset by removing duplicates, converting types, and imputing missing values.

In [3]:
df_clean = df.copy()

duplicates = df_clean.duplicated().sum()
print(f'Duplicate rows found: {duplicates}')
if duplicates > 0:
    df_clean = df_clean.drop_duplicates().reset_index(drop=True)

for col in df_clean.columns:
    if df_clean[col].dtype == 'object':
        coerced = pd.to_numeric(df_clean[col], errors='coerce')
        if coerced.notna().sum() / len(df_clean) > 0.9:
            df_clean[col] = coerced
            print(f'Converted column to numeric: {col}')

missing_before = df_clean.isna().sum().sort_values(ascending=False)
print('Missing values before cleaning:')
print(missing_before[missing_before > 0])

numeric_cols = df_clean.select_dtypes(include=['number']).columns.tolist()
categorical_cols = df_clean.select_dtypes(include=['object', 'category']).columns.tolist()

for col in numeric_cols:
    if df_clean[col].isna().any():
        median_value = df_clean[col].median()
        df_clean[col] = df_clean[col].fillna(median_value)
        print(f'Imputed numeric column {col} with median = {median_value}')

for col in categorical_cols:
    if df_clean[col].isna().any():
        mode_value = df_clean[col].mode().iloc[0] if not df_clean[col].mode().empty else 'Unknown'
        df_clean[col] = df_clean[col].fillna(mode_value)
        print(f'Imputed categorical column {col} with mode = {mode_value}')

print('Missing values after cleaning:')
print(df_clean.isna().sum().sort_values(ascending=False)[lambda x: x > 0])

Duplicate rows found: 0
Missing values before cleaning:
Series([], dtype: int64)
Missing values after cleaning:
Series([], dtype: int64)


## 4. Feature Engineering

Create derived features and prepare numeric and categorical data for modeling.

In [4]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

if 'df_clean' not in globals():
    if 'df' in globals():
        print('Warning: df_clean is not defined. Using original df as df_clean for feature engineering.')
        df_clean = df.copy()
    else:
        raise NameError('df_clean is not defined. Run the data cleaning cell before feature engineering.')

df_features = df_clean.copy()
potential_cols = df_features.columns.tolist()

if 'total_bedrooms' in potential_cols and 'total_rooms' in potential_cols:
    df_features['bedroom_room_ratio'] = df_features['total_bedrooms'] / df_features['total_rooms']
    print('Created feature: bedroom_room_ratio')

if 'population' in potential_cols and 'households' in potential_cols:
    df_features['people_per_household'] = df_features['population'] / df_features['households']
    print('Created feature: people_per_household')

target_candidates = [col for col in df_features.columns if col.lower() in {'median_house_value', 'price', 'house_value', 'sale_price'}]
target_col = target_candidates[0] if target_candidates else None
print(f'Target column detected: {target_col}')

feature_cols = [col for col in df_features.columns if col != target_col]
numeric_features = df_features[feature_cols].select_dtypes(include=['number']).columns.tolist()
categorical_features = df_features[feature_cols].select_dtypes(include=['object', 'category']).columns.tolist()

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features),
], remainder='drop')

print('Numeric features:', numeric_features)
print('Categorical features:', categorical_features)

X_prepared = preprocessor.fit_transform(df_features[feature_cols])
print('Prepared feature matrix shape:', X_prepared.shape)

Target column detected: price
Numeric features: ['area', 'bedrooms', 'bathrooms', 'stories', 'parking']
Categorical features: ['mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'prefarea', 'furnishingstatus']
Prepared feature matrix shape: (545, 20)


## 5. Feature Selection

Use correlation and supervised feature selection to identify the most useful numeric predictors.

In [5]:
from sklearn.feature_selection import SelectKBest, f_regression

if target_col is not None:
    y = df_features[target_col]
    X = df_features[feature_cols].select_dtypes(include=['number']).copy()
    X = X.fillna(X.median())
    selector = SelectKBest(score_func=f_regression, k=min(10, X.shape[1]))
    selector.fit(X, y)
    selected_mask = selector.get_support()
    selected_features = X.columns[selected_mask].tolist()
    feature_scores = pd.Series(selector.scores_, index=X.columns).sort_values(ascending=False)
    print('Top selected numeric features:')
    display(feature_scores.head(15))
    print('Selected features for modeling:', selected_features)
else:
    print('No target column detected. Label the target column before running supervised selection.')

Top selected numeric features:


area         218.884081
bathrooms    198.654521
stories      116.780402
parking       94.143328
bedrooms      84.251022
dtype: float64

Selected features for modeling: ['area', 'bedrooms', 'bathrooms', 'stories', 'parking']


## 6. Model Training, Evaluation, and Saving

Train regression models with multiple algorithms, evaluate them using MAE, RMSE, and R², perform cross-validation and hyperparameter tuning, and save the best pipeline plus charts.

In [6]:
import json
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split, cross_validate, RandomizedSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_selection import SelectKBest, f_regression

charts_dir = Path('Charts')
charts_dir.mkdir(exist_ok=True)

if target_col is None:
    raise ValueError('Target column is required for training.')

y = df_features[target_col]
if 'selected_features' not in globals() or not selected_features:
    X_all = df_features[feature_cols].select_dtypes(include=['number'])
    selector = SelectKBest(score_func=f_regression, k=min(10, X_all.shape[1]))
    selector.fit(X_all, y)
    selected_features = X_all.columns[selector.get_support()].tolist()
    print(f'Fallback selected features: {selected_features}')

selected_features = list(selected_features)
selected_numeric = [c for c in selected_features if c in numeric_features]
selected_categorical = [c for c in selected_features if c in categorical_features]

transformers = []
if selected_numeric:
    transformers.append(('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]), selected_numeric))
if selected_categorical:
    transformers.append(('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ]), selected_categorical))

selected_preprocessor = ColumnTransformer(transformers, remainder='drop') if transformers else 'passthrough'

X = df_features[selected_features].copy()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(random_state=42),
    'XGBoost': XGBRegressor(random_state=42, objective='reg:squarederror', n_jobs=-1)
}
scoring = {
    'rmse': 'neg_root_mean_squared_error',
    'mae': 'neg_mean_absolute_error',
    'r2': 'r2'
}

results = []
pipelines = {}

for name, base_model in models.items():
    pipeline = Pipeline([('preprocessor', selected_preprocessor), ('model', base_model)])
    pipeline.fit(X_train, y_train)
    preds = pipeline.predict(X_test)
    rmse_val = np.sqrt(mean_squared_error(y_test, preds))
    mae_val = mean_absolute_error(y_test, preds)
    r2_val = r2_score(y_test, preds)
    cv_results = cross_validate(pipeline, X, y, cv=5, scoring=scoring, n_jobs=-1)
    metrics = {
        'model': name,
        'rmse': rmse_val,
        'mae': mae_val,
        'r2': r2_val,
        'cv_rmse': -cv_results['test_rmse'].mean(),
        'cv_mae': -cv_results['test_mae'].mean(),
        'cv_r2': cv_results['test_r2'].mean()
    }
    results.append(metrics)
    pipelines[name] = pipeline
    predictions_df = pd.DataFrame({'actual': y_test.reset_index(drop=True), 'predicted': preds})
    predictions_path = charts_dir / f"{name.lower().replace(' ', '_')}_predictions.csv"
    predictions_df.to_csv(predictions_path, index=False)
    print(f'Saved predictions for {name} to {predictions_path}')

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(y_test, preds, alpha=0.6)
    ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
    ax.set_xlabel('Actual values')
    ax.set_ylabel('Predicted values')
    ax.set_title(f'Actual vs Predicted ({name})')
    plt.tight_layout()
    plot_path = charts_dir / f"{name.lower().replace(' ', '_')}_actual_vs_predicted.png"
    fig.savefig(plot_path, dpi=150)
    plt.close(fig)
    print(f'Saved chart for {name} to {plot_path}')

    if hasattr(base_model, 'feature_importances_'):
        importances = base_model.feature_importances_
        fig, ax = plt.subplots(figsize=(10, 6))
        sns.barplot(x=importances, y=selected_features, ax=ax)
        ax.set_title(f'Feature importances ({name})')
        ax.set_xlabel('Importance')
        plt.tight_layout()
        importance_path = charts_dir / f"feature_importance_{name.lower().replace(' ', '_')}.png"
        fig.savefig(importance_path, dpi=150)
        plt.close(fig)
        print(f'Saved feature importance chart for {name} to {importance_path}')

    param_grid = {}
    if name == 'Random Forest':
        param_grid = {
            'model__n_estimators': [100, 200],
            'model__max_depth': [None, 10, 20],
            'model__min_samples_split': [2, 5, 10]
        }
    elif name == 'Gradient Boosting':
        param_grid = {
            'model__n_estimators': [100, 200],
            'model__learning_rate': [0.05, 0.1, 0.2],
            'model__max_depth': [3, 5, 7]
        }
    elif name == 'XGBoost':
        param_grid = {
            'model__n_estimators': [100, 200],
            'model__learning_rate': [0.05, 0.1],
            'model__max_depth': [3, 5, 7]
        }
    if param_grid:
        search = RandomizedSearchCV(
            pipeline,
            param_distributions=param_grid,
            n_iter=6,
            cv=3,
            scoring='neg_root_mean_squared_error',
            n_jobs=-1,
            random_state=42
        )
        search.fit(X_train, y_train)
        tuned_name = name + ' (tuned)'
        tuned_pipeline = search.best_estimator_
        tuned_preds = tuned_pipeline.predict(X_test)
        tuned_rmse = np.sqrt(mean_squared_error(y_test, tuned_preds))
        tuned_mae = mean_absolute_error(y_test, tuned_preds)
        tuned_r2 = r2_score(y_test, tuned_preds)
        tuned_cv = cross_validate(tuned_pipeline, X, y, cv=5, scoring=scoring, n_jobs=-1)
        results.append({
            'model': tuned_name,
            'rmse': tuned_rmse,
            'mae': tuned_mae,
            'r2': tuned_r2,
            'cv_rmse': -tuned_cv['test_rmse'].mean(),
            'cv_mae': -tuned_cv['test_mae'].mean(),
            'cv_r2': tuned_cv['test_r2'].mean()
        })
        pipelines[tuned_name] = tuned_pipeline
        tuned_predictions_path = charts_dir / f"{tuned_name.lower().replace(' ', '_')}_predictions.csv"
        pd.DataFrame({'actual': y_test.reset_index(drop=True), 'predicted': tuned_preds}).to_csv(tuned_predictions_path, index=False)
        print(f'Saved predictions for {tuned_name} to {tuned_predictions_path}')
        fig, ax = plt.subplots(figsize=(8, 6))
        ax.scatter(y_test, tuned_preds, alpha=0.6)
        ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
        ax.set_title(f'Actual vs Predicted ({tuned_name})')
        plt.tight_layout()
        tuned_plot_path = charts_dir / f"{tuned_name.lower().replace(' ', '_')}_actual_vs_predicted.png"
        fig.savefig(tuned_plot_path, dpi=150)
        plt.close(fig)
        print(f'Saved chart for {tuned_name} to {tuned_plot_path}')



Saved predictions for Linear Regression to Charts\linear_regression_predictions.csv
Saved chart for Linear Regression to Charts\linear_regression_actual_vs_predicted.png
Saved predictions for Decision Tree to Charts\decision_tree_predictions.csv
Saved chart for Decision Tree to Charts\decision_tree_actual_vs_predicted.png
Saved feature importance chart for Decision Tree to Charts\feature_importance_decision_tree.png
Saved predictions for Random Forest to Charts\random_forest_predictions.csv
Saved chart for Random Forest to Charts\random_forest_actual_vs_predicted.png
Saved feature importance chart for Random Forest to Charts\feature_importance_random_forest.png
Saved predictions for Random Forest (tuned) to Charts\random_forest_(tuned)_predictions.csv
Saved chart for Random Forest (tuned) to Charts\random_forest_(tuned)_actual_vs_predicted.png
Saved predictions for Gradient Boosting to Charts\gradient_boosting_predictions.csv
Saved chart for Gradient Boosting to Charts\gradient_boostin

In [7]:
metrics_df = pd.DataFrame(results)
metrics_csv = charts_dir / 'model_metrics.csv'
metrics_df.to_csv(metrics_csv, index=False)
print(f'Saved aggregated model metrics to {metrics_csv}')

fig, ax = plt.subplots(figsize=(11, 6))
metrics_df.plot(x='model', y=['rmse', 'mae', 'r2'], kind='bar', ax=ax)
ax.set_title('Model performance comparison')
ax.set_ylabel('Score')
ax.grid(True, axis='y')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
metrics_plot = charts_dir / 'model_performance_comparison.png'
fig.savefig(metrics_plot, dpi=150)
plt.close(fig)
print(f'Saved comparison chart to {metrics_plot}')

best_row = metrics_df.loc[metrics_df['rmse'].idxmin()]
best_model = best_row['model']
best_pipeline = pipelines[best_model]
joblib.dump(best_pipeline, 'best_pipeline.joblib')
print(f'Saved best pipeline as best_pipeline.joblib ({best_model})')

feature_info = {}
for feature in selected_features:
    if feature in selected_numeric:
        feature_info[feature] = {'type': 'numeric'}
    else:
        options = df_features[feature].dropna().unique().tolist()
        feature_info[feature] = {'type': 'categorical', 'options': options[:50]}

with open(charts_dir / 'feature_info.json', 'w', encoding='utf-8') as f:
    json.dump(feature_info, f, indent=2)
print(f"Saved feature metadata to {charts_dir / 'feature_info.json'}")

model_metadata = {
    'best_model': best_model,
    'selected_features': selected_features,
    'train_rows': len(X_train),
    'test_rows': len(X_test)
}
with open(charts_dir / 'model_metadata.json', 'w', encoding='utf-8') as f:
    json.dump(model_metadata, f, indent=2)
print(f"Saved model metadata to {charts_dir / 'model_metadata.json'}")



Saved aggregated model metrics to Charts\model_metrics.csv
Saved comparison chart to Charts\model_performance_comparison.png
Saved best pipeline as best_pipeline.joblib (Linear Regression)
Saved feature metadata to Charts\feature_info.json
Saved model metadata to Charts\model_metadata.json
